# Bidirectional Vehicle-to-Grid (V2G)

GitHub-ready version of the bidirectional smart-charging model.

## 1. Imports

In [ ]:
import os
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional interactive widgets (used in Colab/Jupyter, not required by GitHub)
try:
    import ipywidgets as widgets
    from IPython.display import display
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

# Enable Colab widgets when running in Google Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass


## 2. Data and PV model

In [ ]:
def read_excel_data(filename, sheet_name):
    data = pd.read_excel(filename, sheet_name=sheet_name, header=None)
    return data.values


def find_input_file():
    """Find the Excel input file in Colab, local Jupyter, or the GitHub repo structure."""
    candidates = [
        Path("TP-V2G-Input_data.xlsx"),
        Path("/content/TP-V2G-Input_data.xlsx"),
        Path("../data/TP-V2G-Input_data.xlsx"),
        Path("data/TP-V2G-Input_data.xlsx"),
    ]

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        "TP-V2G-Input_data.xlsx was not found. "
        "Upload it to Colab or place it in the repository data/ folder."
    )


def Mod_PVprod(Ta, E, Ppv_nom):
    """Algebraic photovoltaic production model."""
    NOCT = 48.0
    E_ref = 1000.0
    Tpv_ref = 25.0
    alpha = -4.8e-3

    Tpv = Ta + E * (NOCT - 20.0) / 800.0
    Ppv = (E / E_ref) * (Ppv_nom + alpha * (Tpv - Tpv_ref))

    return np.maximum(Ppv, 0.0)


## 3. Parameters

In [ ]:
# Problem parameters
deg2K = 273.15
kW2W = 1000.0
kWh2Wh = 1000.0
h2min = 60

N_ev = 50
capa_bat = 52          # Battery capacity [kWh]
d_autonomie = 320      # Vehicle range [km]
P1_ch = 22             # Normal charging power [kW]
vmoy = 40 / h2min      # Average speed [km/min]

Ppv_nom = 500e3        # Nominal PV capacity [W]

# Simulation period
j = 174
Nj = 7
ind_0 = j * 24 + 1
ind_f = (j + Nj) * 24 + 1


## 4. Load input data

In [ ]:
filename = find_input_file()

sheet_name = "Demands"
demands = read_excel_data(filename, sheet_name)

Hd = np.array(demands[1 + ind_0:1 + ind_f, 1].tolist()) * kW2W
Pd = np.array(demands[1 + ind_0:1 + ind_f, 2].tolist()) * kW2W
v_tps_h = np.array(demands[1 + ind_0:1 + ind_f, 0].tolist()) * h2min

sheet_name = "Meteo"
meteo = read_excel_data(filename, sheet_name)

E = np.array(meteo[1 + ind_0:1 + ind_f, 1].tolist())
Ta = np.array(meteo[1 + ind_0:1 + ind_f, 2].tolist())

# Interpolate to one-minute time steps
v_tps_mn = np.arange(v_tps_h[0], v_tps_h[-1], 1)

Pd_mn = np.interp(v_tps_mn, v_tps_h, Pd)
E_mn = np.interp(v_tps_mn, v_tps_h, E)
Ta_mn = np.interp(v_tps_mn, v_tps_h, Ta)

# PV production
Ppv = Mod_PVprod(Ta_mn, E_mn, Ppv_nom)

print(f"Loaded {len(v_tps_mn):,} one-minute time steps.")
print(f"Simulation horizon: {Nj} days.")


## 5. Evaluation metrics

In [ ]:
def smoothing_index(P_grid):
    """Load-smoothing indicator J = mean((ΔP_grid)^2)."""
    dP = np.diff(P_grid)
    return float(np.mean(dP ** 2))


def mean_equivalent_cycles(E_throughput_kWh, capa_bat_kWh):
    """Equivalent battery cycles per EV, used as a simple degradation proxy."""
    cycles = E_throughput_kWh / (2.0 * capa_bat_kWh)
    return float(np.mean(cycles))


## 6. Bidirectional simulation

In [ ]:
def simulate_bidirectional(
    cap_factor_weekday=0.5,
    k_need_weekday=0.5,
    soc_reserve=0.30,
    peak_threshold_W=300_000.0,
    seed=174
):
    """
    Simulate bidirectional smart charging.

    Sign convention:
        P_ev > 0  -> charging
        P_ev < 0  -> V2G discharge

    Grid power:
        P_grid = P_d + P_ev - P_pv
    """
    dt_h = 1.0 / 60.0
    soc_min = 0.10
    soc_max = 1.00
    soc_reserve = float(np.clip(soc_reserve, soc_min, soc_max))

    cons_kWh_km = capa_bat / d_autonomie
    Pmax_W = P1_ch * kW2W
    rng = np.random.default_rng(seed)

    T = len(Pd_mn)
    P_ev = np.zeros(T)
    E_throughput = np.zeros(N_ev)

    # All EVs start at 100% SOC.
    E_bat = np.ones(N_ev) * (soc_max * capa_bat)

    t0_midnight = v_tps_mn[0] - 60

    def is_weekend(day_from_start):
        return day_from_start >= 5

    P_base = Pd_mn - Ppv

    for d in range(Nj):
        day_midnight = t0_midnight + d * 1440
        i_day0 = np.searchsorted(v_tps_mn, day_midnight, side="left")
        i_day1 = np.searchsorted(
            v_tps_mn,
            day_midnight + 1440,
            side="left"
        )

        if i_day1 <= i_day0:
            continue

        weekend = is_weekend(d)
        cap_factor = 1.0 if weekend else cap_factor_weekday

        # 1) Mobility and connection windows
        if weekend:
            E_day = np.zeros(N_ev)

            work_arr_idx = np.full(N_ev, i_day0)
            work_dep_idx = np.full(N_ev, i_day1)

            home_arr_idx = np.full(N_ev, i_day0)
            home_dep_idx = np.full(N_ev, i_day1)
        else:
            hdep_home = rng.triangular(
                7.5,
                8.5,
                9.5,
                size=N_ev
            )

            d_traj = rng.triangular(
                10.0,
                30.0,
                50.0,
                size=N_ev
            )

            t_traj_h = d_traj / (vmoy * 60.0)

            harr_work = hdep_home + t_traj_h

            hdep_work = rng.triangular(
                17.0,
                18.0,
                19.0,
                size=N_ev
            )

            hdep_work = np.maximum(
                hdep_work,
                harr_work + 0.25
            )

            harr_home = hdep_work + t_traj_h

            E_day = 2.0 * d_traj * cons_kWh_km

            work_arr_min = (
                day_midnight
                + np.round(harr_work * 60.0).astype(int)
            )

            work_dep_min = (
                day_midnight
                + np.round(hdep_work * 60.0).astype(int)
            )

            work_arr_idx = np.searchsorted(
                v_tps_mn,
                work_arr_min,
                side="left"
            )

            work_dep_idx = np.searchsorted(
                v_tps_mn,
                work_dep_min,
                side="left"
            )

            home_arr_min = (
                day_midnight
                + np.round(harr_home * 60.0).astype(int)
            )

            home_arr_idx = np.searchsorted(
                v_tps_mn,
                home_arr_min,
                side="left"
            )

            hdep_home_next = rng.triangular(
                7.5,
                8.5,
                9.5,
                size=N_ev
            )

            next_midnight = day_midnight + 1440

            home_dep_min_next = (
                next_midnight
                + np.round(hdep_home_next * 60.0).astype(int)
            )

            next_home_dep_idx = np.searchsorted(
                v_tps_mn,
                home_dep_min_next,
                side="left"
            )

            home_dep_idx = np.minimum(
                next_home_dep_idx,
                i_day1
            )

        # Mobility consumption
        E_bat -= E_day
        E_bat = np.clip(
            E_bat,
            soc_min * capa_bat,
            soc_max * capa_bat
        )

        # 2) Charging target
        if weekend:
            E_need = (soc_max * capa_bat) - E_bat
        else:
            E_need = k_need_weekday * E_day

        E_need = np.maximum(E_need, 0.0)
        E_rem = E_need.copy()

        # 3) Minute-by-minute charging and discharging
        for t in range(i_day0, i_day1):
            pv_W = float(max(Ppv[t], 0.0))
            pd_W = float(Pd_mn[t])

            # A) CHARGING: prioritize PV periods
            if pv_W > 0.0:
                P_total_allowed = cap_factor * pv_W

                if P_total_allowed > 0.0:
                    plugged_charge = (
                        (t >= work_arr_idx)
                        & (t < work_dep_idx)
                        & (E_rem > 1e-9)
                        & (
                            E_bat
                            < soc_max * capa_bat - 1e-9
                        )
                    )

                    active = np.where(
                        plugged_charge
                    )[0]

                    if active.size > 0:
                        P_need_W = (
                            E_rem[active] / dt_h
                        ) * 1000.0

                        P_cap_i = np.minimum(
                            Pmax_W,
                            P_need_W
                        )

                        P_alloc = np.zeros(
                            active.size
                        )

                        remaining = P_total_allowed
                        alive = np.ones(
                            active.size,
                            dtype=bool
                        )

                        while (
                            remaining > 1e-6
                            and alive.any()
                        ):
                            idx_local = np.where(
                                alive
                            )[0]

                            share = (
                                remaining
                                / idx_local.size
                            )

                            give = np.minimum(
                                share,
                                P_cap_i[idx_local]
                                - P_alloc[idx_local]
                            )

                            P_alloc[idx_local] += give
                            remaining -= give.sum()

                            alive[idx_local] = (
                                P_alloc[idx_local]
                                < P_cap_i[idx_local]
                                - 1e-9
                            )

                        P_ev[t] += P_alloc.sum()

                        dE = (
                            P_alloc / 1000.0
                        ) * dt_h

                        E_rem[active] = np.maximum(
                            E_rem[active] - dE,
                            0.0
                        )

                        E_bat[active] = np.clip(
                            E_bat[active] + dE,
                            soc_min * capa_bat,
                            soc_max * capa_bat
                        )

                        E_throughput[active] += dE

            # B) DISCHARGING: shave peaks when PV is unavailable
            if (
                pv_W <= 0.0
                and pd_W > peak_threshold_W
            ):
                P_excess = (
                    pd_W - peak_threshold_W
                )

                plugged_dis = (
                    (
                        (t >= work_arr_idx)
                        & (t < work_dep_idx)
                    )
                    |
                    (
                        (t >= home_arr_idx)
                        & (t < home_dep_idx)
                    )
                )

                can_dis = (
                    plugged_dis
                    & (
                        E_bat
                        > soc_reserve * capa_bat
                        + 1e-9
                    )
                )

                active_dis = np.where(
                    can_dis
                )[0]

                if (
                    active_dis.size > 0
                    and P_excess > 0.0
                ):
                    P_total_dis_allowed = min(
                        P_excess,
                        active_dis.size * Pmax_W
                    )

                    E_avail = (
                        E_bat[active_dis]
                        - soc_reserve * capa_bat
                    )

                    P_energy_limit_W = (
                        E_avail / dt_h
                    ) * 1000.0

                    P_cap_dis_i = np.minimum(
                        Pmax_W,
                        P_energy_limit_W
                    )

                    P_cap_dis_i = np.maximum(
                        P_cap_dis_i,
                        0.0
                    )

                    P_alloc_dis = np.zeros(
                        active_dis.size
                    )

                    remaining = (
                        P_total_dis_allowed
                    )

                    alive = (
                        P_cap_dis_i > 1e-9
                    )

                    while (
                        remaining > 1e-6
                        and alive.any()
                    ):
                        idx_local = np.where(
                            alive
                        )[0]

                        share = (
                            remaining
                            / idx_local.size
                        )

                        give = np.minimum(
                            share,
                            P_cap_dis_i[idx_local]
                            - P_alloc_dis[idx_local]
                        )

                        P_alloc_dis[idx_local] += give
                        remaining -= give.sum()

                        alive[idx_local] = (
                            P_alloc_dis[idx_local]
                            < P_cap_dis_i[idx_local]
                            - 1e-9
                        )

                    # Negative sign = discharge to the grid
                    P_ev[t] -= P_alloc_dis.sum()

                    dE_dis = (
                        P_alloc_dis / 1000.0
                    ) * dt_h

                    E_bat[active_dis] = np.clip(
                        E_bat[active_dis] - dE_dis,
                        soc_min * capa_bat,
                        soc_max * capa_bat
                    )

                    E_throughput[active_dis] += (
                        dE_dis
                    )

    P_grid = Pd_mn + P_ev - Ppv

    metrics = {
        "smoothing_index": smoothing_index(
            P_grid
        ),
        "equivalent_cycles_mean": (
            mean_equivalent_cycles(
                E_throughput,
                capa_bat
            )
        ),
        "peak_before_kW": float(
            np.max(P_base) / 1000.0
        ),
        "peak_after_kW": float(
            np.max(P_grid) / 1000.0
        ),
        "energy_charge_kWh": float(
            np.sum(
                np.maximum(P_ev, 0.0)
            )
            / 1000.0
            * dt_h
        ),
        "energy_discharge_kWh": float(
            np.sum(
                np.maximum(-P_ev, 0.0)
            )
            / 1000.0
            * dt_h
        ),
    }

    return P_ev, metrics


## 7. Interactive scenario plot (Colab/Jupyter)

In [ ]:
def plot_scenario_bidirectional(
    cap_factor=0.50,
    k_need=0.50,
    soc_reserve=0.30,
    peak_threshold_kW=300.0
):
    peak_threshold_W = peak_threshold_kW * 1000.0

    P_ev, metrics = simulate_bidirectional(
        cap_factor_weekday=cap_factor,
        k_need_weekday=k_need,
        soc_reserve=soc_reserve,
        peak_threshold_W=peak_threshold_W
    )

    # Correct sign convention from the project report:
    # P_grid = P_d + P_ev - P_pv
    P_grid = Pd_mn + P_ev - Ppv

    time_h = np.arange(len(Pd_mn)) / 60.0

    plt.figure(figsize=(12, 5))
    plt.plot(
        time_h,
        Pd_mn / 1000.0,
        label="Base demand"
    )
    plt.plot(
        time_h,
        Ppv / 1000.0,
        label="PV generation"
    )
    plt.plot(
        time_h,
        P_ev / 1000.0,
        label="EV power (signed)"
    )
    plt.plot(
        time_h,
        P_grid / 1000.0,
        label="Grid power"
    )
    plt.axhline(
        peak_threshold_kW,
        linestyle="--",
        label="Peak threshold"
    )

    plt.title(
        "Bidirectional Vehicle-to-Grid Peak Shaving"
    )
    plt.xlabel("Time from start [h]")
    plt.ylabel("Power [kW]")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("---- Metrics ----")
    for key, value in metrics.items():
        print(f"{key}: {value}")


In [ ]:
if WIDGETS_AVAILABLE:
    cap_slider = widgets.FloatSlider(
        value=0.50,
        min=0.00,
        max=1.50,
        step=0.05,
        description="cap_factor",
        continuous_update=False,
    )

    kneed_slider = widgets.FloatSlider(
        value=0.50,
        min=0.00,
        max=1.00,
        step=0.05,
        description="k_need",
        continuous_update=False,
    )

    socres_slider = widgets.FloatSlider(
        value=0.30,
        min=0.10,
        max=1.00,
        step=0.05,
        description="soc_reserve",
        continuous_update=False,
    )

    peak_slider = widgets.FloatSlider(
        value=300.0,
        min=0.0,
        max=600.0,
        step=10.0,
        description="peak_kW",
        continuous_update=False,
    )

    ui = widgets.VBox(
        [
            cap_slider,
            kneed_slider,
            socres_slider,
            peak_slider,
        ]
    )

    out = widgets.interactive_output(
        plot_scenario_bidirectional,
        {
            "cap_factor": cap_slider,
            "k_need": kneed_slider,
            "soc_reserve": socres_slider,
            "peak_threshold_kW": peak_slider,
        },
    )

    display(ui, out)
else:
    print("ipywidgets is not installed. The static GitHub figure below will still work.")


## 8. Static GitHub figure

In [ ]:
# Static GitHub-friendly V2G scenario
CAP_FACTOR = 0.50
K_NEED = 0.50
SOC_RESERVE = 0.30
PEAK_THRESHOLD_KW = 300.0

P_ev_v2g, metrics_v2g = simulate_bidirectional(
    cap_factor_weekday=CAP_FACTOR,
    k_need_weekday=K_NEED,
    soc_reserve=SOC_RESERVE,
    peak_threshold_W=PEAK_THRESHOLD_KW * 1000.0
)

P_grid_v2g = Pd_mn + P_ev_v2g - Ppv
time_h = np.arange(len(Pd_mn)) / 60.0

plt.figure(figsize=(12, 5))
plt.plot(
    time_h,
    Pd_mn / 1000.0,
    label="Base demand"
)
plt.plot(
    time_h,
    Ppv / 1000.0,
    label="PV generation"
)
plt.plot(
    time_h,
    P_ev_v2g / 1000.0,
    label="EV power (signed)"
)
plt.plot(
    time_h,
    P_grid_v2g / 1000.0,
    label="Grid power"
)
plt.axhline(
    PEAK_THRESHOLD_KW,
    linestyle="--",
    label="Peak threshold"
)

plt.title(
    "Bidirectional Vehicle-to-Grid Peak Shaving"
)
plt.xlabel("Time from start [h]")
plt.ylabel("Power [kW]")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

figure_dir = Path("figures")
figure_dir.mkdir(exist_ok=True)

figure_path = (
    figure_dir
    / "03_bidirectional_v2g_peak_shaving.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print(f"Saved figure: {figure_path}")
print("---- V2G scenario metrics ----")
for key, value in metrics_v2g.items():
    print(f"{key}: {value}")
